# Stage 1 Training — Cross-Embodiment Shared Latent Space (Multi-Robot)

Yan et al. (2026) — shared E_h / E_X / D_X + per-robot E_r / D_r.

**EIGENGRASP_ONLINE sampling:** robot poses are sampled live from the eigengrasp
PCA basis (committed in the repo) + a MuJoCo collision filter. No precomputed
valid_poses caches, no big DONG cache in RAM.

**Drive files needed (`MyDrive/AIST-hand/`):**

| File | Description |
|------|-------------|
| `hograspnet_abl14_r6.csv` | Human poses, 6D repr |
| `data/processed/hagrid_dong_r6.csv` | HaGRID extra human anchors (open/fist) |
| `dex-urdf/` | Shadow URDF + kinematics |

> Eigengrasp basis + MJCF for both robots ship in the cloned repo
> (`robot/hands/*/datasets/processed/eigengrasp*.npz`, `third_party/mujoco_menagerie/`).
> Online sampling keeps RAM low, so Shadow + Allegro train together on a free T4.

**Runtime:** GPU — `Runtime > Change runtime type > T4 GPU`

## 0. Check GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode == 0:
    print(result.stdout)
else:
    print('No GPU found. Go to Runtime > Change runtime type and select T4 GPU.')

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Config

In [ ]:
import os
from pathlib import Path

DRIVE_ROOT   = Path('/content/drive/MyDrive/AIST-hand')
GITHUB_TOKEN = ''
GITHUB_USER  = 'isapedraza'
REPO_NAME    = 'AIST-hand'

# ── Robot config ──────────────────────────────────────────────────────────────
# EIGENGRASP_ONLINE sampling: eigengrasp basis + MJCF are read from the cloned
# repo (see _ROBOT_STATIC below). No precomputed valid_poses, no big DONG cache
# in RAM -> both robots fit easily. The loader hard-drops valid_poses when an
# eigengrasp basis is set, so online sampling is guaranteed.
ROBOTS = {
    'shadow':  {'zero_wrj': True},
    'allegro': {'zero_wrj': False},
}

# ── Human data ────────────────────────────────────────────────────────────────
HUMAN_ROT_REPR    = 'r6'
CSV_PATH          = DRIVE_ROOT / 'hograspnet_abl14_r6.csv'
EXTRA_HUMAN_CSV   = DRIVE_ROOT / 'data/processed/hagrid_dong_r6.csv'
EXTRA_HUMAN_RATIO = 0.10

# ── Checkpoint ────────────────────────────────────────────────────────────────
CKPT_PATH   = DRIVE_ROOT / 'checkpoints/stage1_r6_latest.pt'
RESUME_CKPT = None   # set to CKPT_PATH to resume

# ── Training hyperparams ──────────────────────────────────────────────────────
B              = 50_000
N_STEPS        = 6_000
LOG_EVERY      = 50
CKPT_EVERY     = 500
VAL_EVERY      = 500
N_EVAL_BATCHES = 20
B_EVAL         = 5000
SEED           = -1

Z_DIM      = 16
SHARED_DIM = 1024
LR         = 1e-3
LAMBDA_C   = 10.0
LAMBDA_REC = 5.0
LAMBDA_LTC = 1.0
LAMBDA_TMP = 0.1
W_R        = 1.0
W_JOINTS   = 1.2
W_AHG      = 0.07
N_TRIPLETS = None
MARGIN     = 0.1

print(f'Drive root : {DRIVE_ROOT}')
print(f'Robots     : {list(ROBOTS.keys())}')
print(f'B={B}  N_STEPS={N_STEPS}  VAL_EVERY={VAL_EVERY}')

## 3. Clone repo from GitHub

In [ ]:
REPO_DIR = f'/content/{REPO_NAME}'
BRANCH   = 'main'

if not os.path.exists(REPO_DIR):
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git'
    os.system(f'git clone {clone_url} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} fetch origin')

os.system(f'git -C {REPO_DIR} checkout {BRANCH}')
os.system(f'git -C {REPO_DIR} pull origin {BRANCH}')
print(f'Checked out branch: {BRANCH}')

REPO_ROOT = Path(REPO_DIR)
DEX_ROOT  = DRIVE_ROOT / 'dex-urdf'
print(f'Repo    : {REPO_ROOT}')
print(f'dex-urdf: {DEX_ROOT}')

# Per-robot URDF + hand_config (resolved after clone)
_ROBOT_STATIC = {
    'shadow': {
        'urdf':        DEX_ROOT  / 'robots/hands/shadow_hand/shadow_hand_right.urdf',
        'hand_config': REPO_ROOT / 'robot/hand-configs/shadow_hand_right.yaml',
        'eigengrasp':  REPO_ROOT / 'robot/hands/shadow_hand/datasets/processed/dexonomy_shadow_eigengrasps_balanced_sample.npz',
        'mjcf':        REPO_ROOT / 'third_party/mujoco_menagerie/shadow_hand/right_hand.xml',
    },
    'allegro': {
        'urdf':        REPO_ROOT / 'robot/hands/allegro_hand/allegro_hand_right.urdf',
        'hand_config': REPO_ROOT / 'robot/hand-configs/allegro_hand_right.yaml',
        'eigengrasp':  REPO_ROOT / 'robot/hands/allegro_hand/datasets/processed/eigengrasp_allegro.npz',
        'mjcf':        REPO_ROOT / 'third_party/mujoco_menagerie/wonik_allegro/right_hand.xml',
    },
}
for name, cfg in ROBOTS.items():
    cfg.update(_ROBOT_STATIC[name])

## 4. Install dependencies

In [ ]:
import torch
print(f'torch={torch.__version__}  cuda={torch.version.cuda}')

# setuptools-scm required by latent-retargeting pyproject.toml
!pip install -q setuptools-scm
!pip install -q pytorch-kinematics
!pip install -q -e /content/AIST-hand/models/latent-retargeting
!pip install -q -e /content/AIST-hand/human

print('Done.')

## 5. Validate Drive files + write robots YAML

In [ ]:
import yaml, psutil

ram = psutil.virtual_memory()
print(f'RAM total : {ram.total / 1e9:.1f} GB')
print(f'RAM avail : {ram.available / 1e9:.1f} GB')
print(f'RAM used  : {ram.used / 1e9:.1f} GB')

required = [CSV_PATH, EXTRA_HUMAN_CSV]
for name, cfg in ROBOTS.items():
    required += [cfg['eigengrasp'], cfg['mjcf'], cfg['urdf'], cfg['hand_config']]

missing = [str(p) for p in required if not Path(p).exists()]
if missing:
    raise FileNotFoundError('Missing files:\n' + '\n'.join(missing))
print('All required files found.')

robots_yaml_path = Path('/tmp/robots_config.yaml')
robots_yaml_data = {
    name: {
        'urdf':        str(cfg['urdf']),
        'hand_config': str(cfg['hand_config']),
        'eigengrasp':  str(cfg['eigengrasp']),
        'mjcf':        str(cfg['mjcf']),
        'zero_wrj':    cfg['zero_wrj'],
    }
    for name, cfg in ROBOTS.items()
}
robots_yaml_path.write_text(yaml.dump(robots_yaml_data))
print(f'robots_config written: {robots_yaml_path}')
print(yaml.dump(robots_yaml_data))

## 6. Run Stage 1 training

In [ ]:
import subprocess, sys

print(f'HUMAN_ROT_REPR : {HUMAN_ROT_REPR}')
print(f'CSV            : {CSV_PATH}')
print(f'CKPT           : {CKPT_PATH}')
print(f'RESUME_CKPT    : {RESUME_CKPT}')
print(f'Robots         : {list(ROBOTS.keys())}')
print(f'Z_DIM={Z_DIM}  SHARED_DIM={SHARED_DIM}  MARGIN={MARGIN}')
print(f'SEED           : {SEED} ({"random" if SEED < 0 else "fixed"})')

cmd = [
    sys.executable, '-u',
    str(REPO_ROOT / 'models/latent-retargeting/scripts/train_cross_emb.py'),
    '--repo_root',         str(REPO_ROOT),
    '--dex_root',          str(DEX_ROOT),
    '--csv_path',          str(CSV_PATH),
    '--ckpt_path',         str(CKPT_PATH),
    '--robots_config',     str(robots_yaml_path),
    '--extra_human_csv',   str(EXTRA_HUMAN_CSV),
    '--extra_human_ratio', str(EXTRA_HUMAN_RATIO),
    '--human_rot_repr',    HUMAN_ROT_REPR,
    '--b',                 str(B),
    '--n_steps',           str(N_STEPS),
    '--log_every',         str(LOG_EVERY),
    '--ckpt_every',        str(CKPT_EVERY),
    '--z_dim',             str(Z_DIM),
    '--shared_dim',        str(SHARED_DIM),
    '--lr',                str(LR),
    '--lambda_c',          str(LAMBDA_C),
    '--lambda_rec',        str(LAMBDA_REC),
    '--lambda_ltc',        str(LAMBDA_LTC),
    '--lambda_tmp',        str(LAMBDA_TMP),
    '--w_r',               str(W_R),
    '--w_joints',          str(W_JOINTS),
    '--w_ahg',             str(W_AHG),
    '--margin',            str(MARGIN),
    '--seed',              str(SEED),
    '--val_every',         str(VAL_EVERY),
    '--n_eval_batches',    str(N_EVAL_BATCHES),
    '--b_eval',            str(B_EVAL),
    '--log_metric_stats',
    '--skip_final_eval',
]
if N_TRIPLETS is not None:
    cmd.extend(['--n_triplets', str(N_TRIPLETS)])
if RESUME_CKPT is not None:
    cmd.extend(['--resume_ckpt', str(RESUME_CKPT)])

print('\nTraining command:')
print(' '.join(cmd))

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError(f'Training failed with code {proc.returncode}')

## 7. Memory diagnostics

In [ ]:
import torch, psutil

allocated = torch.cuda.memory_allocated() / 1e9
reserved  = torch.cuda.memory_reserved()  / 1e9
total_gpu = torch.cuda.get_device_properties(0).total_memory / 1e9
ram       = psutil.virtual_memory()

print(f'GPU allocated : {allocated:.2f} GB')
print(f'GPU reserved  : {reserved:.2f} GB')
print(f'GPU total     : {total_gpu:.2f} GB')
print(f'GPU free      : {total_gpu - reserved:.2f} GB')
print(f'RAM used      : {ram.used / 1e9:.1f} GB / {ram.total / 1e9:.1f} GB')